## Confirmatory run analysis

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT=Path("..")
RESULTS=ROOT/"results"/"week12"/"confirmatory"
ANALYSIS=RESULTS/"analysis"
ANALYSIS.mkdir(exist_ok=True)

POWER_PATH=RESULTS/"primary_power_runs.csv"
NULL_PATH=RESULTS/"formal_clean_null_runs.csv"
MANIFEST_PATH=RESULTS/"confirmatory_execution_manifest.csv"

print("Results:",RESULTS.resolve())

Results: /Users/jessica/fidelity-metric-lab/results/week12/confirmatory


In [3]:
# load formal outputs
def read_formal_csv(path):
    return pd.read_csv(
        path,
        keep_default_na=False,
        na_values=[""]
    )

power=read_formal_csv(POWER_PATH)
null=read_formal_csv(NULL_PATH)
manifest=read_formal_csv(MANIFEST_PATH)

print("Power rows:",len(power))
print("Null rows :",len(null))
print("Total     :",len(power)+len(null))
print("Manifest  :",len(manifest))

Power rows: 22000
Null rows : 8000
Total     : 30000
Manifest  : 30000


### Final integrity check 

In [4]:
def check_basic(df,name):
    assert df["run_id"].is_unique,f"{name}: duplicate run_id"
    assert df["p_value"].notna().all(),f"{name}: missing p-values"
    assert np.isfinite(df["p_value"]).all(),f"{name}: non-finite p-values"
    assert df["p_value"].between(0,1).all(),f"{name}: invalid p-values"
    assert df["reject"].isin([0,1]).all(),f"{name}: invalid reject values"
    assert np.isfinite(df["runtime"]).all(),f"{name}: invalid runtimes"

check_basic(power,"POWER")
check_basic(null,"NULL")

assert len(power)==22000
assert len(null)==8000
assert len(power)+len(null)==30000

assert power["kind"].eq("power").all()
assert null["kind"].eq("null").all()

assert set(power["N"].unique())=={50,200}
assert set(null["N"].unique())=={50,200}

assert (power["N"].value_counts().to_dict()=={
    50:12000,
    200:10000
})

assert (null["N"].value_counts().to_dict()=={
    50:4000,
    200:4000
})

power_cells=(
    power.groupby(
        ["scenario","severity","N","test"],
        dropna=False
    )
    .size()
)

null_cells=(
    null.groupby(
        ["representation","N","test"],
        dropna=False
    )
    .size()
)

assert len(power_cells)==110
assert len(null_cells)==40

assert (power_cells==200).all()
assert (null_cells==200).all()

# each formal cell must contain reps 0...199
expected_reps=set(range(200))

for _,g in power.groupby(
    ["scenario","severity","N","test"]
):
    assert set(g["rep"])==expected_reps

for _,g in null.groupby(
    ["representation","N","test"]
):
    assert set(g["rep"])==expected_reps

# exact agreement with frozen manifest
result_ids=set(power["run_id"])|set(null["run_id"])
manifest_ids=set(manifest["run_id"])

assert result_ids==manifest_ids

print("FINAL INTEGRITY CHECK PASSED")
print("Power cells:",len(power_cells))
print("Null cells :",len(null_cells))
print("Formal runs:",len(result_ids))

FINAL INTEGRITY CHECK PASSED
Power cells: 110
Null cells : 40
Formal runs: 30000


In [5]:
# save integrity record 
integrity={
    "power_rows":int(len(power)),
    "null_rows":int(len(null)),
    "total_rows":int(len(power)+len(null)),
    "power_cells":int(len(power_cells)),
    "null_cells":int(len(null_cells)),
    "repetitions_per_cell":200,
    "duplicate_run_ids":0,
    "missing_manifest_runs":0,
    "integrity_passed":True
}

with open(
    ANALYSIS/"integrity_report.json",
    "w"
) as f:
    json.dump(integrity,f,indent=2)

integrity

{'power_rows': 22000,
 'null_rows': 8000,
 'total_rows': 30000,
 'power_cells': 110,
 'null_cells': 40,
 'repetitions_per_cell': 200,
 'duplicate_run_ids': 0,
 'missing_manifest_runs': 0,
 'integrity_passed': True}

In [6]:
### 95% Confidence Interval helper
def wilson_ci(k,n,z=1.959963984540054):
    k=float(k)
    n=float(n)

    p=k/n
    z2=z**2

    denom=1+z2/n

    center=(
        p+z2/(2*n)
    )/denom

    half=(
        z*np.sqrt(
            p*(1-p)/n+
            z2/(4*n**2)
        )
    )/denom

    return center-half,center+half

In [7]:
### Formal type-1 error 
type1=(
    null.groupby(
        ["representation","N","test"],
        as_index=False
    )
    .agg(
        rejections=("reject","sum"),
        n_reps=("reject","size"),
        mean_p_value=("p_value","mean"),
        median_p_value=("p_value","median"),
        mean_runtime=("runtime","mean")
    )
)

type1["type_I_error"]=(
    type1["rejections"]/
    type1["n_reps"]
)

cis=type1.apply(
    lambda r:wilson_ci(
        r["rejections"],
        r["n_reps"]
    ),
    axis=1
)

type1["wilson_low"]=[x[0] for x in cis]
type1["wilson_high"]=[x[1] for x in cis]

# preregistered calibration rule:
# 4–16 rejections inclusive out of 200
type1["calibration_status"]=np.where(
    type1["rejections"].between(4,16),
    "within preregistered range",
    "FLAG"
)

type1=type1.sort_values(
    ["representation","N","test"]
).reset_index(drop=True)

type1

,representation,N,test,rejections,n_reps,mean_p_value,median_p_value,mean_runtime,type_I_error,wilson_low,wilson_high,calibration_status
0,Broken Timing Structural,50,C2ST-Linear,6,200,0.569554,0.574257,0.013339,0.030,0.013820,0.063894,within preregistered range
1,Broken Timing Structural,50,C2ST-RBF,3,200,0.602030,0.643564,0.007560,0.015,0.005114,0.043166,FLAG
2,Broken Timing Structural,50,Energy,10,200,0.518465,0.534653,0.018782,0.050,0.027383,0.089578,within preregistered range
3,Broken Timing Structural,50,MMD,11,200,0.517624,0.544554,0.019441,0.055,0.030985,0.095787,within preregistered range
4,Broken Timing Structural,50,Sinkhorn,7,200,0.497525,0.504950,0.733128,0.035,0.017056,0.070471,within preregistered range
5,Broken Timing Structural,200,C2ST-Linear,5,200,0.572970,0.633663,0.015776,0.025,0.010725,0.057178,within preregistered range
6,Broken Timing Structural,200,C2ST-RBF,4,200,0.569059,0.594059,0.013711,0.020,0.007804,0.050287,within preregistered range
7,Broken Timing Structural,200,Energy,9,200,0.521089,0.544554,0.105753,0.045,0.023853,0.083297,within preregistered range
8,Broken Timing Structural,200,MMD,9,200,0.520792,0.529703,0.105559,0.045,0.023853,0.083297,within preregistered range
9,Broken Timing Structural,200,Sinkhorn,8,200,0.541139,0.579208,1.068375,0.040,0.020406,0.076932,within preregistered range


In [8]:
### Show only calibration problems
type1_flags=type1[
    type1["calibration_status"]=="FLAG"
].copy()

print(
    "Flagged null configurations:",
    len(type1_flags)
)

type1_flags

Flagged null configurations: 4


,representation,N,test,rejections,n_reps,mean_p_value,median_p_value,mean_runtime,type_I_error,wilson_low,wilson_high,calibration_status
1,Broken Timing Structural,50,C2ST-RBF,3,200,0.602030,0.643564,0.007560,0.015,0.005114,0.043166,FLAG
21,Message MiniLM,50,C2ST-RBF,1,200,0.603713,0.658416,0.009118,0.005,0.000883,0.027774,FLAG
35,Template TF-IDF,200,C2ST-Linear,2,200,0.583960,0.574257,0.054292,0.010,0.002747,0.035722,FLAG
36,Template TF-IDF,200,C2ST-RBF,2,200,0.589802,0.599010,0.374726,0.010,0.002747,0.035722,FLAG


The four flags are due to under-rejection, not an excess of false positives:
- Broken Timing, N=50, C2ST-RBF: 3/200 = 0.015.
- Message MiniLM, N=50, C2ST-RBF: 1/200 = 0.005.
- Template TF-IDF, N=200, C2ST-Linear: 2/200 = 0.010.
- Template TF-IDF, N=200, C2ST-RBF: 2/200 = 0.010.
This preliminarily indicates that these C2ST configurations are conservative, rather than anti-conservative. We are neither correcting nor removing them; instead, any corresponding power results should be interpreted with this caveat in mind.

In [9]:
# save type-1 table
type1.to_csv(
    ANALYSIS/"type1_wilson.csv",
    index=False
)

type1_flags.to_csv(
    ANALYSIS/"type1_flags.csv",
    index=False
)

### Formal power and 95% confidence interval

In [10]:
power_summary=(
    power.groupby(
        ["scenario","representation","severity","N","test"],
        as_index=False
    )
    .agg(
        rejections=("reject","sum"),
        n_reps=("reject","size"),
        mean_p_value=("p_value","mean"),
        median_p_value=("p_value","median"),
        mean_runtime=("runtime","mean")
    )
)

power_summary["power"]=(
    power_summary["rejections"]/
    power_summary["n_reps"]
)

cis=power_summary.apply(
    lambda r:wilson_ci(
        r["rejections"],
        r["n_reps"]
    ),
    axis=1
)

power_summary["wilson_low"]=[
    x[0] for x in cis
]

power_summary["wilson_high"]=[
    x[1] for x in cis
]

power_summary=power_summary.sort_values(
    ["scenario","severity","N","test"]
).reset_index(drop=True)

power_summary

,scenario,representation,severity,N,test,rejections,n_reps,mean_p_value,median_p_value,mean_runtime,power,wilson_low,wilson_high
0,Broken Timing,Broken Timing Structural,0.1,50,Energy,77,200,0.137228,0.079208,0.011633,0.385,0.320333,0.454001
1,Broken Timing,Broken Timing Structural,0.1,50,MMD,75,200,0.144505,0.079208,0.012218,0.375,0.310854,0.443857
2,Broken Timing,Broken Timing Structural,0.1,50,Sinkhorn,10,200,0.740644,1.000000,0.575511,0.050,0.027383,0.089578
3,Broken Timing,Broken Timing Structural,0.1,200,C2ST-Linear,67,200,0.144752,0.089109,0.009364,0.335,0.273241,0.402978
4,Broken Timing,Broken Timing Structural,0.1,200,C2ST-RBF,67,200,0.148267,0.089109,0.009712,0.335,0.273241,0.402978
...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,Template Repetition,Template TF-IDF,0.6,50,C2ST-Linear,200,200,0.010050,0.009901,0.019286,1.000,0.981155,1.000000
106,Template Repetition,Template TF-IDF,0.6,50,C2ST-RBF,200,200,0.010000,0.009901,0.020130,1.000,0.981155,1.000000
107,Template Repetition,Template TF-IDF,0.6,50,Energy,200,200,0.009901,0.009901,0.021686,1.000,0.981155,1.000000
108,Template Repetition,Template TF-IDF,0.6,50,MMD,200,200,0.009901,0.009901,0.024068,1.000,0.981155,1.000000


In [11]:
# attach matching null calibration 
calibration=type1[
    [
        "representation",
        "N",
        "test",
        "type_I_error",
        "wilson_low",
        "wilson_high",
        "calibration_status"
    ]
].rename(columns={
    "wilson_low":"type1_wilson_low",
    "wilson_high":"type1_wilson_high"
})

power_calibrated=power_summary.merge(
    calibration,
    on=[
        "representation",
        "N",
        "test"
    ],
    how="left",
    validate="many_to_one"
)

assert len(power_calibrated)==110
assert power_calibrated[
    "type_I_error"
].notna().all()

power_calibrated

,scenario,representation,severity,N,test,rejections,n_reps,mean_p_value,median_p_value,mean_runtime,power,wilson_low,wilson_high,type_I_error,type1_wilson_low,type1_wilson_high,calibration_status
0,Broken Timing,Broken Timing Structural,0.1,50,Energy,77,200,0.137228,0.079208,0.011633,0.385,0.320333,0.454001,0.050,0.027383,0.089578,within preregistered range
1,Broken Timing,Broken Timing Structural,0.1,50,MMD,75,200,0.144505,0.079208,0.012218,0.375,0.310854,0.443857,0.055,0.030985,0.095787,within preregistered range
2,Broken Timing,Broken Timing Structural,0.1,50,Sinkhorn,10,200,0.740644,1.000000,0.575511,0.050,0.027383,0.089578,0.035,0.017056,0.070471,within preregistered range
3,Broken Timing,Broken Timing Structural,0.1,200,C2ST-Linear,67,200,0.144752,0.089109,0.009364,0.335,0.273241,0.402978,0.025,0.010725,0.057178,within preregistered range
4,Broken Timing,Broken Timing Structural,0.1,200,C2ST-RBF,67,200,0.148267,0.089109,0.009712,0.335,0.273241,0.402978,0.020,0.007804,0.050287,within preregistered range
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,Template Repetition,Template TF-IDF,0.6,50,C2ST-Linear,200,200,0.010050,0.009901,0.019286,1.000,0.981155,1.000000,0.020,0.007804,0.050287,within preregistered range
106,Template Repetition,Template TF-IDF,0.6,50,C2ST-RBF,200,200,0.010000,0.009901,0.020130,1.000,0.981155,1.000000,0.025,0.010725,0.057178,within preregistered range
107,Template Repetition,Template TF-IDF,0.6,50,Energy,200,200,0.009901,0.009901,0.021686,1.000,0.981155,1.000000,0.040,0.020406,0.076932,within preregistered range
108,Template Repetition,Template TF-IDF,0.6,50,MMD,200,200,0.009901,0.009901,0.024068,1.000,0.981155,1.000000,0.035,0.017056,0.070471,within preregistered range


In [12]:
# save final primary tables 
power_summary.to_csv(
    ANALYSIS/"power_wilson.csv",
    index=False
)

power_calibrated.to_csv(
    ANALYSIS/"power_with_null_calibration.csv",
    index=False
)

print("Saved:")
for p in sorted(ANALYSIS.glob("*")):
    print("-",p.name)

Saved:
- integrity_report.json
- power_wilson.csv
- power_with_null_calibration.csv
- type1_flags.csv
- type1_wilson.csv


### Primary analysis overview

In [13]:
overview=power_calibrated[
    [
        "scenario",
        "severity",
        "N",
        "test",
        "power",
        "wilson_low",
        "wilson_high",
        "type_I_error",
        "calibration_status"
    ]]

overview

,scenario,severity,N,test,power,wilson_low,wilson_high,type_I_error,calibration_status
0,Broken Timing,0.1,50,Energy,0.385,0.320333,0.454001,0.050,within preregistered range
1,Broken Timing,0.1,50,MMD,0.375,0.310854,0.443857,0.055,within preregistered range
2,Broken Timing,0.1,50,Sinkhorn,0.050,0.027383,0.089578,0.035,within preregistered range
3,Broken Timing,0.1,200,C2ST-Linear,0.335,0.273241,0.402978,0.025,within preregistered range
4,Broken Timing,0.1,200,C2ST-RBF,0.335,0.273241,0.402978,0.020,within preregistered range
...,...,...,...,...,...,...,...,...,...
105,Template Repetition,0.6,50,C2ST-Linear,1.000,0.981155,1.000000,0.020,within preregistered range
106,Template Repetition,0.6,50,C2ST-RBF,1.000,0.981155,1.000000,0.025,within preregistered range
107,Template Repetition,0.6,50,Energy,1.000,0.981155,1.000000,0.040,within preregistered range
108,Template Repetition,0.6,50,MMD,1.000,0.981155,1.000000,0.035,within preregistered range


In [14]:
pd.set_option("display.max_rows",200)

overview.round(3)

,scenario,severity,N,test,power,wilson_low,wilson_high,type_I_error,calibration_status
0,Broken Timing,0.1,50,Energy,0.385,0.320,0.454,0.050,within preregistered range
1,Broken Timing,0.1,50,MMD,0.375,0.311,0.444,0.055,within preregistered range
2,Broken Timing,0.1,50,Sinkhorn,0.050,0.027,0.090,0.035,within preregistered range
3,Broken Timing,0.1,200,C2ST-Linear,0.335,0.273,0.403,0.025,within preregistered range
4,Broken Timing,0.1,200,C2ST-RBF,0.335,0.273,0.403,0.020,within preregistered range
5,Broken Timing,0.1,200,Energy,1.000,0.981,1.000,0.045,within preregistered range
6,Broken Timing,0.1,200,MMD,1.000,0.981,1.000,0.045,within preregistered range
7,Broken Timing,0.1,200,Sinkhorn,0.415,0.349,0.484,0.040,within preregistered range
8,Broken Timing,0.3,50,C2ST-Linear,0.265,0.209,0.330,0.030,within preregistered range
9,Broken Timing,0.3,50,C2ST-RBF,0.235,0.182,0.298,0.015,FLAG


### Complete type-1 matrix

In [15]:
type1_matrix=type1.pivot(
    index=["representation","N"],
    columns="test",
    values="type_I_error"
)

type1_matrix.round(3)

test                          C2ST-Linear  C2ST-RBF  Energy    MMD  Sinkhorn
representation           N                                                  
Broken Timing Structural 50         0.030     0.015   0.050  0.055     0.035
                         200        0.025     0.020   0.045  0.045     0.040
Campaign MiniLM          50         0.035     0.025   0.060  0.060     0.055
                         200        0.050     0.060   0.065  0.065     0.075
Message MiniLM           50         0.020     0.005   0.030  0.030     0.025
                         200        0.035     0.030   0.030  0.035     0.065
Template TF-IDF          50         0.020     0.025   0.040  0.035     0.040
                         200        0.010     0.010   0.030  0.030     0.065

In [16]:
# range check
print("Within range:",
      (type1["calibration_status"]=="within preregistered range").sum())

print("Flagged:",
      (type1["calibration_status"]=="FLAG").sum())

print("Total:",
      len(type1))

Within range: 36
Flagged: 4
Total: 40


### Confirmatory power 

In [17]:
# n = 50
power50_matrix=(
    power_calibrated[
        power_calibrated["N"]==50
    ]
    .pivot(
        index=["scenario","severity"],
        columns="test",
        values="power"
    )
)

power50_matrix.round(3)

test                                      C2ST-Linear  C2ST-RBF  Energy  \
scenario                        severity                                  
Broken Timing                   0.1               NaN       NaN   0.385   
                                0.3             0.265     0.235   1.000   
                                0.6             0.905     0.895   1.000   
Entity Inconsistency            0.1               NaN       NaN   0.185   
                                0.3               NaN       NaN   0.875   
                                0.6             0.025     0.015   1.000   
Missing Cross-Channel Reference 0.1               NaN       NaN   0.050   
                                0.3             0.035     0.015   0.190   
                                0.6             0.210     0.170   0.710   
Over-Polished Language          0.1               NaN       NaN   0.195   
                                0.3             0.025     0.000   0.975   
                                0.6             0.330     0.235   1.000   
Template Repetition             0.1             0.035     0.040   0.370   
                                0.3             0.860     0.870   1.000   
                                0.6             1.000     1.000   1.000   

test                                        MMD  Sinkhorn  
scenario                        severity                   
Broken Timing                   0.1       0.375     0.050  
                                0.3       1.000     0.280  
                                0.6       1.000     0.535  
Entity Inconsistency            0.1       0.175       NaN  
                                0.3       0.880       NaN  
                                0.6       1.000       NaN  
Missing Cross-Channel Reference 0.1       0.050     0.025  
                                0.3       0.200     0.085  
                                0.6       0.715     0.245  
Over-Polished Language          0.1       0.190     0.010  
                                0.3       0.975       NaN  
                                0.6       1.000       NaN  
Template Repetition             0.1       0.380     0.365  
                                0.3       1.000     1.000  
                                0.6       1.000     1.000

In [18]:
# n=200
power200_matrix=(
    power_calibrated[
        power_calibrated["N"]==200
    ]
    .pivot(
        index=["scenario","severity"],
        columns="test",
        values="power"
    )
)

power200_matrix.round(3)

test                                      C2ST-Linear  C2ST-RBF  Energy  \
scenario                        severity                                  
Broken Timing                   0.1             0.335     0.335   1.000   
                                0.3             1.000     1.000     NaN   
Entity Inconsistency            0.1             0.000     0.000   0.995   
                                0.3             0.125     0.145     NaN   
                                0.6             0.800     0.675     NaN   
Missing Cross-Channel Reference 0.1             0.045     0.030   0.540   
                                0.3             0.315     0.390   1.000   
                                0.6             0.805     0.920   1.000   
Over-Polished Language          0.1             0.055     0.020   1.000   
                                0.3             0.845     0.680     NaN   
                                0.6             1.000     0.995     NaN   
Template Repetition             0.1             0.965     0.975   1.000   

test                                        MMD  Sinkhorn  
scenario                        severity                   
Broken Timing                   0.1       1.000     0.415  
                                0.3         NaN     1.000  
Entity Inconsistency            0.1       0.995     0.080  
                                0.3         NaN     0.165  
                                0.6         NaN     0.540  
Missing Cross-Channel Reference 0.1       0.535     0.605  
                                0.3       1.000     0.890  
                                0.6       1.000     1.000  
Over-Polished Language          0.1       1.000     0.460  
                                0.3         NaN     0.005  
                                0.6         NaN     0.155  
Template Repetition             0.1       1.000     1.000

The main takeaway is quite clear. Energy and MMD are, by far, the two most consistent and sensitive configurations. Across almost all defect types, their performance trends are very similar; furthermore, with N=200, they detect changes that were quite difficult to identify with N=50. This strongly reinforces an observation we made in Week 7: both families respond in an almost parallel fashion.

C2ST continues to exhibit the "blind spots" we set out to investigate. We now have stronger evidence that these cannot be explained merely by the use of a linear classifier, as an RBF kernel also fails to eliminate them in a general sense. The most striking example is "Entity Inconsistency": with N=200 and a 10% severity level, Energy and MMD reach 0.995, whereas both the Linear and RBF-based approaches yield 0.000. In this same scenario, the "Campaign MiniLM" calibrations fall within an acceptable range; therefore, this result cannot simply be dismissed as a Type I calibration issue. It represents a genuine blind spot for these C2ST configurations under this specific representation.

"Broken Timing" also clearly illustrates the effect of $N$. At a 10% severity level, with $N=50$, the Energy/MMD scores are around 0.38, whereas with $N=200$ they reach 1.00. C2ST rises to 0.335, but remains well below that level. This serves as a very clean demonstration of the interaction between sample size and test configuration.

"Missing Cross-Channel" is arguably one of the most useful scenarios for the paper, as it shows a gradual progression. For instance, with $N=200$, the Energy/MMD scores increase from approximately 0.54 to 1.00 as the prevalence rises, while C2ST and Sinkhorn respond more slowly. Here, the benefit of increasing $N$ is clearly evident.

"Template Repetition" becomes extremely easy as $N$ increases. With $N=200$ and only 10% prevalence, the scores for all configurations reach approximately 0.965–1.00. This is interesting because it contrasts sharply with semantic defects: the same sample size does not benefit all types of defects equally.

"Over-Polished" retains one of the most intriguing failure modes. Energy and MMD quickly reach 1.00, while Sinkhorn continues to show very weak—and even non-monotonic—behavior: with $N=200$, the scores are approximately 0.460, 0.005, and 0.155 as the severity increases. This is precisely the kind of result we should not attempt to "fix"; it constitutes scientific insight into how the paired-Sinkhorn configuration responds to this specific defect.

Regarding Type I errors: 36 of the 40 configurations fell within the pre-registered range. The four that fell outside this range did so because they were too conservative, not because they were generating an excessive number of false positives: They rejected H0 fewer times than expected under the clean-vs-clean scenario. With $\alpha=0.05$, we would expect around 10 rejections out of 200; those four configurations yielded 1–3 rejections. This reduces false positives, but it can also make the power appear lower than what a perfectly calibrated test would exhibit. Consequently, they do not invalidate the experiment. However, we will add a note of caution to any power metrics associated with those four configurations.

Most importantly, the overall pattern of results does not appear to depend on those four flags. For instance, the major blind spots for "Entity" occur in the Campaign MiniLM setting, where C2ST is correctly calibrated.